# 4.0.4/6 Clustering Integrated

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(tidyverse)
library(dittoSeq)
library(stringr)
library(enrichR)
library(purrr)
library(here)
library(glue)
}))

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
set.seed(0)

In [ ]:
args <- commandArgs(TRUE)
op <- args[2]
print(op)

## Load Data

In [ ]:
#integrated_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.3-integrated_harmony_object_{op}.rds")))
integrated_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.5-integrated_filt.rds")))

In [ ]:
DefaultAssay(integrated_obj) <- "RNA"

## Clustering

In [ ]:
# Define resolutions
resolutions <- seq(from =  0.05, to =  0.5, by =  0.05)
  
# Find clusters
resolutions_dfs <- lapply(resolutions, function(x) {
  integrated_obj <- FindClusters(integrated_obj, resolution = x,
                                     graph.name = "wsnn")
  df <- data.frame(
    barcode = colnames(integrated_obj),
    cluster_id = as.character(integrated_obj$seurat_clusters)
  )
  df
})

# Merge data frames
names(resolutions_dfs) <- resolutions
resolutions_df <- dplyr::bind_rows(resolutions_dfs, .id = "resolution")
resolutions_df$resolution <- as.numeric(resolutions_df$resolution)

In [ ]:
list_plots <- lapply(unique(resolutions_df$resolution), function(x) {
  p <- add_reso_results(integrated_obj, resolutions_df, resolution=x)
  umap <- DimPlot(p, 
                  reduction = "wnn.umap", 
                  raster=FALSE,
                  group.by="seurat_clusters",
                  label = T,
                  label.box = TRUE,
                  label.size = 3,
                  repel = FALSE) + NoLegend()+
    labs(title=glue::glue("Resolution {x}"))
  return(umap)
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
integrated_obj <- FindClusters(integrated_obj, resolution = 0.05,
                                     graph.name = "wsnn")

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)
p1 <- DimPlot(
    integrated_obj,
      reduction = "wnn.umap",
      raster=FALSE,
      label = T,
      label.box = TRUE,
      label.size = 4,
      repel = FALSE) + NoLegend() +
    ggtitle('Clustering') + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))
p1

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15, warn=-1,verbose = FALSE)
VlnPlot(integrated_obj, feature=c("nCount_RNA", "nFeature_RNA",'S.Score',
                                "G2M.Score", "pct_ribosomal", "pct_mt"), 
        ncol=2, pt.size = 0)

In [ ]:
ggplotColours <- function(n = 6, h = c(0, 360) + 15){
  if ((diff(h) %% 360) < 1) h[2] <- h[2] - 360/n
  hcl(h = (seq(h[1], h[2], length = n)), c = 100, l = 65)
}

color_list <- ggplotColours(n=14)

In [ ]:
names(color_list) <- levels(integrated_obj)

In [ ]:
# custom order
integrated_obj$patient <- factor(integrated_obj$patient,
                                 levels=c("p19", "p15", "p16", "p17","p14","p9","p4",
                                          "p1","p10","p5","p6","p3","p13","p7","p8","p2"))

In [ ]:
rel <- dittoBarPlot(integrated_obj, "ident", group.by = "patient", 
                    color.panel = color_list,
                    retain.factor.levels = TRUE) + 
    labs(title = "Relative proportion of \n clusters cells per patient") + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
          text =  element_text(hjust = 0.5,size = 14),
          axis.text.x = element_text(size = 12, color="black",angle = 45,  vjust = 1, hjust = 1),
          axis.text.y = element_text(size = 12, color="black"))

abs <- dittoBarPlot(integrated_obj, "ident", group.by = "patient", 
                    color.panel = color_list,
                    scale = "count", 
                    retain.factor.levels = TRUE) + 
    labs(title ="Absolute proportion of \n clusters cells per patient") + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
          text =  element_text(hjust = 0.5,size = 14),
          axis.text.x = element_text(size = 12, color="black",angle = 45,  vjust = 1, hjust = 1),
          axis.text.y = element_text(size = 12, color="black")) + NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
abs+rel

In [ ]:
rel <- dittoBarPlot(integrated_obj,"patient" , group.by = "ident", 
                     color.panel = colors_patients, 
                     x.reorder = c(1,2,3,6,7,8,9,10,11,13,14,4,5,12),
                     retain.factor.levels = TRUE) + 
    labs(title = "Proportion patient's\ncells per cluster") + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
          text =  element_text(hjust = 0.5,size = 14),
          axis.text.x = element_text(size = 12, color="black",angle = 45,  vjust = 1, hjust = 1),
          axis.text.y = element_text(size = 12, color="black"))

abs <- dittoBarPlot(integrated_obj, "patient", group.by = "ident", 
             scale = "count", 
             color.panel = colors_patients, 
             retain.factor.levels = TRUE) + 
    labs(title ="Absolute proportion of \n patient cells per cluster") + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
          text =  element_text(hjust = 0.5,size = 14),
          axis.text.x = element_text(size = 12, color="black",angle = 45,  vjust = 1, hjust = 1),
          axis.text.y = element_text(size = 12, color="black")) + NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
abs+rel

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 10)
p2 <- DimPlot(
  integrated_obj,
   reduction = "wnn.umap",
   group.by = "patient",
   pt.size = 0.1,
   cols = colors_patients,
   raster=FALSE
) + NoLegend() + ggtitle('Patient') + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold")) 

p3 <- DimPlot(
  integrated_obj,
   reduction = "wnn.umap",
   group.by = "cell_groups",
   pt.size = 0.1,
   cols=colors_cellg,
   raster=FALSE
) + ggtitle('Cell Groups') + 
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))

#patchwork::wrap_plots(list(p3,p2,p1,rel), ncol=2)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)
patchwork::wrap_plots(list(list_plots[[5]],list_plots[[3]],list_plots[[1]], rel), ncol=4)

In [ ]:
grid_integr <- patchwork::wrap_plots(list(list_plots[[1]],list_plots[[3]],list_plots[[5]], rel), ncol=4)
ggplot2::ggsave(
  filename = here(glue("{int}/{plt_dir}/4.0.6-intgr.svg")),
  plot = grid_integr,
  width = 20,
  height = 5,
  dpi = 500,
  device = "svg"
)
ggplot2::ggsave(
  filename = here(glue("{int}/{plt_dir}/4.0.6-intgr.png")),
  plot = grid_integr,
  width = 20,
  height = 5,
  dpi = 500,
  device = "png"
)

In [ ]:
rna.markers <- FindAllMarkers(integrated_obj, max.cells.per.ident = 1000, 
                              logfc.threshold = 0.1, only.pos = TRUE)

In [ ]:
rna.markers <- rna.markers %>%
    dplyr::filter(p_val_adj < 0.05) %>%
    arrange(desc("avg_log2FC"))

In [ ]:
write_csv(rna.markers, here(glue("{int}/{robj_dir}/4.0.6-{op}_harmony_markers.csv")))

#### Cluster 0

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
VlnPlot(integrated_obj, ident=0, group.by="patient", ncol=2, pt.size=0,
        feature=c("COL11A1", "SEMA3A", "EPHA5","MEST","SEMA3E","SOX4","ROBO2","CNTN4"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
VlnPlot(integrated_obj, ident=0, group.by="patient", ncol=2, pt.size=0,
        feature=c("CALR", "MEOX2", "MDK","HMGB1","PPIA","ROBO1"))

#### Cluster 1

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
VlnPlot(integrated_obj, ident=1, group.by="patient", ncol=2, pt.size=0,
        feature=c("ADGRV1", "OTX2", "MITF","NOTCH2","MARK1","SOX4","RMST","AFF3"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 15)
VlnPlot(integrated_obj, ident=1, group.by="patient", ncol=2, pt.size=0,
        feature=c("ARID1B", "MSI2", "PTPRG","MARK1","FOXP1","NT5C2","FGFR2"))

### EnrichR

In [ ]:
dbs <- c('WikiPathways_2024_Human', 
    'Reactome_2022', 
    'MSigDB_Hallmark_2020',
    'KEGG_2021_Human')

In [ ]:
get_db_suffix <- function(db_name) {
  case_when(
    db_name == 'MSigDB_Hallmark_2020' ~ " HMK",
    db_name == 'KEGG_2021_Human' ~ "KEGG",
    db_name == 'Reactome_2022' ~ "",
    db_name == 'WikiPathways_2024_Human' ~ "",
    TRUE ~ ""
  )
}

In [ ]:
plot_cluster_enrichment <- function(all_res, cluster_id) {

  # Top 10 but sorted by p-value adjusted and combined score
  top_terms <- all_res %>%
    filter(Adjusted.P.value < 0.05) %>% 
    arrange(Adjusted.P.value, desc(Combined.Score)) %>%
    head(10) %>%
    mutate(Term_Wrapped = str_wrap(Term_Suffix, width = 50)) %>%
    mutate(Term_Ordered = factor(Term_Wrapped, levels = unique(Term_Wrapped[order(Combined.Score)])))

   power_trans <- scales::trans_new("power_0.7", 
                                   transform = function(x) x^0.7, 
                                   inverse = function(x) x^(1/0.7))
                                    
  p <- ggplot(top_terms, aes(x = Combined.Score, y = Term_Ordered)) +
    geom_vline(xintercept = seq(0, max(top_terms$Combined.Score), length.out = 5), 
               linetype = "dotted", color = "gray60", alpha = 0.6) +
    geom_point(aes(size = gene_count, fill = Adjusted.P.value), 
               shape = 21,    
               color = "black", 
               stroke = 0.8,  
               alpha = 0.9) +
    scale_fill_viridis_c(option = "C", direction = -1, 
                         limits = c(0, 0.05), oob = scales::squish,
                         name = "Adjusted\nP-value") +
    scale_size_continuous(
      range = c(3, 15), 
      trans = power_trans,
      limits = c(1, 200),
      breaks = c(1, 5, 20, 50, 100, 200),
      name = "# Gene Overlap"
    ) +
    scale_x_continuous(expand = expansion(mult = 0.2)) + 
    coord_cartesian(clip = "off") + 
    
    labs(title = paste("Cluster", cluster_id),
         x = "Combined Score", 
         y = NULL) +
    theme_classic(base_size = 14) +
    theme(
      plot.title = element_text(face = "bold", size = 18, hjust = 0.5),
      axis.title.x = element_text(face = "bold",size = 16),
      axis.text.y = element_text(size = 18, face = "plain", color = "black", lineheight = 0.8),
      axis.text.x = element_text(size = 14,color = "black"),
      legend.position = "right",
      legend.key.height = unit(1, "cm"), 
      legend.title = element_text(face = "bold")
    )

  return(p)
}

In [ ]:
target_clusters <- c(0, 1)

res_list <- lapply(target_clusters, function(cl) {
  # filter
  df_tmp <- rna.markers %>%
    filter(cluster == cl) %>%
    filter(p_val_adj < 0.05 & avg_log2FC > 0.25) %>%
    arrange(desc(avg_log2FC)) 
  
  gene_list <- df_tmp$gene
  
  # Run EnrichR
  if (length(gene_list) > 0) {
    message(paste("Running EnrichR for Cluster", cl, "with", length(gene_list), "genes..."))
    
    res <- enrichr(gene_list, databases = dbs)
    # combine dataframe result list
    all_res <- map_dfr(names(res), function(db) {
      df <- res[[db]]
      if (nrow(df) > 0) {
        df$Database <- db
        df$Term_Suffix <- paste0(df$Term, get_db_suffix(db))
        return(df)
      } else {
        return(NULL)
      }
    })
  
    if (nrow(all_res) == 0) {
      message(paste("No enrichment found for Cluster", cl))
      return(NULL)
    }
    # Extract gene count 
    all_res <- all_res %>%
      mutate(
        gene_count = as.numeric(sapply(str_split(Overlap, "/"), `[`, 1)),
        gene_ratio = as.numeric(sapply(str_split(Overlap, "/"), `[`, 1)) / 
                   as.numeric(sapply(str_split(Overlap, "/"), `[`, 2)),
        cluster = cl
    )
      
    p <- plot_cluster_enrichment(all_res, cl)
    
    if (!is.null(p)) {
     return(list("plot" = p,
                 "data" = all_res
                ))
    }
  }
})

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)

cp <- cowplot::plot_grid(plotlist = list(res_list[1][[1]]$plot, res_list[2][[1]]$plot),
                   align = "hv",
                   axis = "trbl",
                   ncol = 1,
                   nrow = 2)
cp

In [ ]:
#options(repr.plot.width = 6, repr.plot.height = 5, warn=-1,verbose = FALSE)
#integrated_obj$patient <- factor(integrated_obj$patient,
#                                 levels=c("p19", "p17", "p16","p15", "p14","p13",
#                                          "p9","p4","p1","p10","p5","p6","p3","p7","p8","p2"))

#VlnPlot(integrated_obj, feature=c("pct_ribosomal"), 
#        ncol=2, pt.size = 0, group.by="patient", cols=colors_patients)

In [ ]:
ggplot2::ggsave(
  filename = here(glue("{int}/{plt_dir}/4.0.6-Enrichments.svg")),
  plot = cp,
  width = 12,
  height = 12,
  dpi = 500,
  device = "svg")

In [ ]:
ora_res <- dplyr::bind_rows(res_list[1][[1]]$data, res_list[2][[1]]$data)

In [ ]:
#write_csv(ora_res, here(glue("{int}/{robj_dir}/4.0.4-ORA_cluster_markers.csv")))
write_csv(ora_res, here(glue("{int}/{robj_dir}/4.0.6-ORA_cluster_markers.csv")))

In [ ]:
#saveRDS(integrated_obj, here(glue("{int}/{robj_dir}/4.0.4-intgr_clustering.rds")))
saveRDS(integrated_obj, here(glue("{int}/{robj_dir}/4.0.6-intgr_clustering.rds")))

## Session

In [ ]:
sessionInfo()